# Moroccan Sign Language Translation (SLT)

**Pipeline:** OpenPose + RGB → Multimodal Fusion → LLM (Qwen2.5-3B / JAIS-13B + LoRA) → Arabic Translation

Dataset format expected:
```python
{
    "pose_sequence": openpose_keypoints,  # (T, 150) float32
    "rgb_frames":    rgb_frames,          # (T, H, W, 3) uint8
    "target_text":   annotation           # str, Arabic
}
```

**No synthetic text. No chatbot. No hallucinations.**  
The model learns exclusively from real dataset annotations.

---
| Section | Content |
|---|---|
| 1 | Setup environment |
| 2 | Dataset loading |
| 3 | OpenPose visualization |
| 4 | RGB visualization |
| 5 | Skeleton encoder |
| 6 | RGB encoder |
| 7 | Multimodal fusion |
| 8 | LoRA integration |
| 9 | Qwen2.5-3B training |
| 10 | JAIS-13B training |
| 11 | BLEU / ROUGE / METEOR / BERTScore |
| 12 | Inference pipeline |
| 13 | Translation generation |
| 14 | Educational template formatting |
| 15 | Checkpoint saving |
| 16 | Ablation study |
| 17 | Visualization plots |

## 1. Setup Environment

In [ ]:
# Install dependencies not already present in the environment.
# Run once; restart kernel if packages were freshly installed.
import subprocess, sys

pkgs = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "accelerate>=0.29.0",
    "bitsandbytes>=0.43.0",
    "datasets>=2.18.0",
    "evaluate>=0.4.1",
    "sacrebleu>=2.3.1",
    "rouge-score>=0.1.2",
    "nltk>=3.8.1",
    "bert-score>=0.3.13",
    "arabic-reshaper>=3.0.0",
    "python-bidi>=0.4.2",
    "matplotlib>=3.8.0",
    "tqdm>=4.66.0",
    "opencv-python-headless>=4.9.0.80",
    "Pillow>=10.2.0",
    "scipy>=1.12.0",
    "einops>=0.7.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print('Dependencies ready.')

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import unicodedata
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Arabic text rendering
import arabic_reshaper
from bidi.algorithm import get_display

# HuggingFace stack
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings('ignore')

# ── Repo root on sys.path so mosl.* imports work ──────────────────────────
REPO_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# Fallback: detect by presence of mosl/
for _p in [Path('.'), Path('..'), Path('/workspaces/Master_Multimodal-Moroccan-SLG-main')]:
    if (_p / 'mosl').is_dir():
        REPO_ROOT = _p.resolve()
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── Reuse existing mosl modules ───────────────────────────────────────────
from mosl.data.dataset import MoSLSkelsDataset, mosl_collate, COORDS_PER_FRAME
from mosl.text.tokenizer import WordTokenizer
from mosl.render.pose_bridge import (
    load_clip, interpolate_gaps, compute_transform, draw_pose_frame, _apply,
    COCO18_LIMBS, COCO18_COLORS, HAND_EDGES, CONF_THR,
)
from mosl.render.temporal import TemporalConfig

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR        = REPO_ROOT / 'data'
PROCESSED_DIR   = DATA_DIR / 'processed'
KEYPOINTS_DIR   = PROCESSED_DIR / 'keypoints_2d'
OPENPOSE_DIR    = PROCESSED_DIR / 'openpose_json'
FINAL_DATA_DIR  = REPO_ROOT / 'third_party' / 'Prompt2Sign' / 'tools' / '2D_to_3D' / 'final_data'
VOCAB_PATH      = PROCESSED_DIR / 'vocab.json'
LABELS_CSV      = DATA_DIR / 'labels.csv'
OUTPUTS_DIR     = REPO_ROOT / 'outputs' / 'slt'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Repo   : {REPO_ROOT}')
print(f'Outputs: {OUTPUTS_DIR}')

## 2. Dataset Loading

In [ ]:
# ── SLT Dataset ───────────────────────────────────────────────────────────
# Wraps the existing MoSLSkelsDataset and adds RGB frame loading.
# Each sample exposes the format required by the SLT pipeline:
#   { pose_sequence, rgb_frames, target_text }

class SLTDataset(Dataset):
    """Multimodal SLT dataset: OpenPose keypoints + RGB frames + Arabic annotation.

    Falls back gracefully when RGB frames are unavailable (returns zeros).
    All annotations come exclusively from the real dataset labels — no synthetic text.
    """

    def __init__(
        self,
        mode: str,
        tokenizer: WordTokenizer,
        final_data_dir: Optional[Path] = None,
        video_root: Optional[Path] = None,
        max_rgb_frames: int = 16,
        rgb_size: int = 224,
        repo_root: Optional[Path] = None,
    ) -> None:
        self.mode = mode
        self.tokenizer = tokenizer
        self.max_rgb_frames = max_rgb_frames
        self.rgb_size = rgb_size

        repo_root = repo_root or REPO_ROOT
        final_data_dir = final_data_dir or FINAL_DATA_DIR
        self.video_root = video_root or (DATA_DIR / 'raw' / 'vedios-dataset')

        # Reuse the existing skeleton dataset for pose + text
        self._skels_ds = MoSLSkelsDataset(
            mode, tokenizer=tokenizer,
            final_data_dir=final_data_dir,
            repo_root=repo_root,
        )

    def __len__(self) -> int:
        return len(self._skels_ds)

    def _load_rgb(self, clip_id: str) -> np.ndarray:
        """Load uniformly-sampled RGB frames for a clip.
        Returns (T, H, W, 3) uint8 array, or zeros if video not found.
        """
        # Search for the video under video_root
        candidates = list(self.video_root.rglob(f'{clip_id}.mp4'))
        if not candidates:
            return np.zeros((self.max_rgb_frames, self.rgb_size, self.rgb_size, 3), dtype=np.uint8)

        cap = cv2.VideoCapture(str(candidates[0]))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return np.zeros((self.max_rgb_frames, self.rgb_size, self.rgb_size, 3), dtype=np.uint8)

        indices = np.linspace(0, total - 1, self.max_rgb_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                frame = np.zeros((self.rgb_size, self.rgb_size, 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (self.rgb_size, self.rgb_size))
            frames.append(frame)
        cap.release()
        return np.stack(frames, axis=0)  # (T, H, W, 3)

    def __getitem__(self, idx: int) -> dict:
        skel_sample = self._skels_ds[idx]
        clip_id = skel_sample['clip_id']

        # Decode the annotation from token ids (strip specials)
        target_text = self.tokenizer.decode(
            skel_sample['text_ids'].tolist(), strip_specials=True
        )

        rgb = self._load_rgb(clip_id)  # (T, H, W, 3) uint8

        return {
            'pose_sequence': skel_sample['pose'],          # (T, 150) float32
            'rgb_frames':    torch.from_numpy(rgb),        # (T, H, W, 3) uint8
            'target_text':   target_text,                  # str, Arabic annotation
            # Keep auxiliary fields for training
            'text_ids':      skel_sample['text_ids'],
            'time':          skel_sample['time'],
            'n_frames':      skel_sample['n_frames'],
            'clip_id':       clip_id,
        }


def slt_collate(batch: list[dict]) -> dict:
    """Collate SLT samples: pad pose + time; stack RGB; collect text strings."""
    B = len(batch)
    T_max = max(s['n_frames'] for s in batch)
    T_rgb = batch[0]['rgb_frames'].shape[0]
    H = batch[0]['rgb_frames'].shape[1]
    W = batch[0]['rgb_frames'].shape[2]

    pose     = torch.zeros(B, T_max, COORDS_PER_FRAME)
    time_t   = torch.zeros(B, T_max)
    pose_mask = torch.zeros(B, T_max, dtype=torch.bool)
    rgb      = torch.zeros(B, T_rgb, H, W, 3, dtype=torch.uint8)
    n_frames = torch.empty(B, dtype=torch.long)
    texts, clip_ids = [], []

    for b, s in enumerate(batch):
        T = s['n_frames']
        pose[b, :T]      = s['pose_sequence']
        time_t[b, :T]    = s['time']
        pose_mask[b, :T] = True
        rgb[b]           = s['rgb_frames']
        n_frames[b]      = T
        texts.append(s['target_text'])
        clip_ids.append(s['clip_id'])

    return {
        'pose':       pose,
        'pose_mask':  pose_mask,
        'time':       time_t,
        'rgb_frames': rgb,
        'n_frames':   n_frames,
        'target_text': texts,
        'clip_ids':   clip_ids,
    }


# ── Load tokenizer and datasets ───────────────────────────────────────────
if VOCAB_PATH.exists():
    tokenizer_word = WordTokenizer.load(VOCAB_PATH)
    print(f'Vocabulary: {tokenizer_word.vocab_size} tokens ({tokenizer_word.n_signs} signs)')
else:
    print(f'[WARN] vocab.json not found at {VOCAB_PATH}.')
    print('       Run: python -m mosl.text.tokenizer data/labels.csv')
    tokenizer_word = None

datasets_available = FINAL_DATA_DIR.exists() if FINAL_DATA_DIR else False

if tokenizer_word and datasets_available:
    train_ds = SLTDataset('train', tokenizer=tokenizer_word)
    dev_ds   = SLTDataset('dev',   tokenizer=tokenizer_word)
    test_ds  = SLTDataset('test',  tokenizer=tokenizer_word)
    print(f'Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}')
    sample = train_ds[0]
    print(f'Sample clip_id   : {sample["clip_id"]}')
    print(f'  pose_sequence  : {tuple(sample["pose_sequence"].shape)}')
    print(f'  rgb_frames     : {tuple(sample["rgb_frames"].shape)}')
    print(f'  target_text    : {sample["target_text"]!r}')
else:
    print('[INFO] Dataset files not found — demo mode with synthetic shapes.')
    print('       Expected:', FINAL_DATA_DIR)
    train_ds = dev_ds = test_ds = None
    sample = {
        'pose_sequence': torch.zeros(60, 150),
        'rgb_frames':    torch.zeros(16, 224, 224, 3, dtype=torch.uint8),
        'target_text':   'أَنَا',
        'clip_id':       'demo_clip',
        'n_frames':      60,
        'time':          torch.linspace(0.01, 1.0, 60),
        'text_ids':      torch.tensor([1, 4, 2]),
    }

## 3. OpenPose Visualization

In [ ]:
# ── OpenPose skeleton visualization ──────────────────────────────────────
# Reuses mosl.render.pose_bridge drawing primitives.
# Draws COCO-18 body + 21-point hands on a black canvas.

def _ar(text: str) -> str:
    """Reshape + bidi-reorder Arabic string for matplotlib."""
    return get_display(arabic_reshaper.reshape(text))


def visualize_openpose_sequence(
    pose_seq: torch.Tensor,   # (T, 150) — 50 joints × (x, y, z)
    title: str = '',
    n_frames: int = 6,
    canvas: int = 256,
) -> None:
    """Plot a grid of skeleton frames from a pose sequence.

    The 150-dim vector is interpreted as 50 joints × (x, y, z).
    We project to 2D (x, y) and synthesise a confidence of 1.0 for
    non-zero joints so the existing draw_pose_frame function can be reused.
    """
    T = pose_seq.shape[0]
    indices = np.linspace(0, T - 1, min(n_frames, T), dtype=int)

    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3.5))
    if len(indices) == 1:
        axes = [axes]

    for ax, t in zip(axes, indices):
        frame = pose_seq[t].numpy()  # (150,)
        joints = frame.reshape(50, 3)  # (50, x/y/z)

        # Build COCO-18 body array (18, 3) from first 18 joints
        body_xy = joints[:18, :2]  # (18, 2)
        # Normalise to [0, canvas]
        xy_min, xy_max = body_xy.min(), body_xy.max()
        span = max(xy_max - xy_min, 1e-6)
        body_xy = (body_xy - xy_min) / span * (canvas * 0.8) + canvas * 0.1
        conf = (joints[:18, :2].sum(axis=1) != 0).astype(np.float32)
        body = np.concatenate([body_xy, conf[:, None]], axis=1)  # (18, 3)

        # Hands: joints 18-38 (left) and 39-59 (right)
        def _hand(start: int) -> np.ndarray:
            h = joints[start:start + 21, :2].copy()
            h = (h - xy_min) / span * (canvas * 0.8) + canvas * 0.1
            c = (joints[start:start + 21, :2].sum(axis=1) != 0).astype(np.float32)
            return np.concatenate([h, c[:, None]], axis=1)

        hand_l = _hand(18) if joints.shape[0] >= 39 else np.zeros((21, 3))
        hand_r = _hand(39) if joints.shape[0] >= 60 else np.zeros((21, 3))

        img = draw_pose_frame(
            body, hand_l, hand_r,
            np.zeros((0, 3)), canvas, draw_face=False,
        )
        ax.imshow(np.array(img))
        ax.set_title(f't={t}', fontsize=9)
        ax.axis('off')

    if title:
        fig.suptitle(_ar(title), fontsize=12, fontfamily='DejaVu Sans')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'openpose_visualization.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUTS_DIR / "openpose_visualization.png"}')


visualize_openpose_sequence(
    sample['pose_sequence'],
    title=sample['target_text'],
    n_frames=6,
)

## 4. RGB Visualization

In [ ]:
# ── RGB frame visualization ───────────────────────────────────────────────

def visualize_rgb_frames(
    rgb_frames: torch.Tensor,  # (T, H, W, 3) uint8
    title: str = '',
    n_frames: int = 8,
) -> None:
    T = rgb_frames.shape[0]
    indices = np.linspace(0, T - 1, min(n_frames, T), dtype=int)
    frames_np = rgb_frames.numpy()

    fig, axes = plt.subplots(1, len(indices), figsize=(2.5 * len(indices), 3))
    if len(indices) == 1:
        axes = [axes]

    for ax, t in zip(axes, indices):
        ax.imshow(frames_np[t])
        ax.set_title(f't={t}', fontsize=9)
        ax.axis('off')

    if title:
        fig.suptitle(_ar(title), fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'rgb_visualization.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUTS_DIR / "rgb_visualization.png"}')


visualize_rgb_frames(
    sample['rgb_frames'],
    title=sample['target_text'],
    n_frames=8,
)

## 5. Skeleton Encoder

In [ ]:
# ── Skeleton Encoder ──────────────────────────────────────────────────────
# Temporal transformer that encodes the OpenPose keypoint sequence.
# Input : (B, T, 150)  — 50 joints × (x, y, z) per frame
# Output: (B, d_model) — pooled clip-level representation

class SkeletonEncoder(nn.Module):
    """Encode a variable-length pose sequence to a fixed-size embedding.

    Architecture:
      Linear(150 → d_model) → sinusoidal PE → N-layer Transformer Encoder
      → mean-pool over real frames → LayerNorm
    """

    def __init__(
        self,
        pose_dim: int = 150,
        d_model: int = 512,
        nhead: int = 8,
        n_layers: int = 4,
        d_ff: int = 2048,
        dropout: float = 0.1,
        max_len: int = 512,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(pose_dim, d_model)

        # Sinusoidal positional encoding (Vaswani 2017)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, activation='relu', batch_first=True, norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self,
        pose: torch.Tensor,       # (B, T, 150)
        pose_mask: torch.Tensor,  # (B, T) bool — True for real frames
    ) -> torch.Tensor:            # (B, d_model)
        x = self.input_proj(pose) * math.sqrt(self.d_model)  # (B, T, d)
        T = x.size(1)
        x = x + self.pe[:, :T, :]  # add positional encoding
        # PyTorch: src_key_padding_mask is True where padding
        x = self.encoder(x, src_key_padding_mask=~pose_mask)  # (B, T, d)
        # Mean-pool over real frames
        mask_f = pose_mask.unsqueeze(-1).float()  # (B, T, 1)
        pooled = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)  # (B, d)
        return self.norm(pooled)


# Smoke test
skel_enc = SkeletonEncoder().to(DEVICE)
n_params = sum(p.numel() for p in skel_enc.parameters())
print(f'SkeletonEncoder: {n_params:,} parameters')

_pose  = sample['pose_sequence'].unsqueeze(0).to(DEVICE)          # (1, T, 150)
_pmask = torch.ones(1, _pose.shape[1], dtype=torch.bool).to(DEVICE)
_skel_out = skel_enc(_pose, _pmask)
print(f'Output shape: {tuple(_skel_out.shape)}')  # (1, 512)

## 6. RGB Encoder

In [ ]:
# ── RGB Encoder ───────────────────────────────────────────────────────────
# Lightweight CNN + temporal pooling for RGB frame sequences.
# Input : (B, T, H, W, 3) uint8
# Output: (B, d_model)

class RGBEncoder(nn.Module):
    """Encode a sequence of RGB frames to a fixed-size embedding.

    Architecture:
      MobileNetV3-Small backbone (pretrained on ImageNet, frozen by default)
      → per-frame features (B*T, 576) → Linear(576 → d_model)
      → reshape (B, T, d_model) → mean-pool → LayerNorm

    Using MobileNetV3-Small keeps the parameter count low and avoids
    downloading large weights in constrained environments.
    """

    def __init__(
        self,
        d_model: int = 512,
        freeze_backbone: bool = True,
    ) -> None:
        super().__init__()
        import torchvision.models as tvm
        backbone = tvm.mobilenet_v3_small(weights='IMAGENET1K_V1')
        # Remove the classifier head; keep features up to adaptive pool
        self.features = backbone.features
        self.pool = backbone.avgpool
        feat_dim = 576  # MobileNetV3-Small output channels

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad_(False)

        self.proj = nn.Linear(feat_dim, d_model)
        self.norm = nn.LayerNorm(d_model)

        # ImageNet normalisation constants
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _preprocess(self, rgb: torch.Tensor) -> torch.Tensor:
        """(B*T, H, W, 3) uint8 → (B*T, 3, H, W) float32 normalised."""
        x = rgb.float() / 255.0                    # (B*T, H, W, 3)
        x = x.permute(0, 3, 1, 2)                  # (B*T, 3, H, W)
        return (x - self.mean) / self.std

    def forward(
        self,
        rgb_frames: torch.Tensor,  # (B, T, H, W, 3) uint8
    ) -> torch.Tensor:             # (B, d_model)
        B, T, H, W, C = rgb_frames.shape
        flat = rgb_frames.view(B * T, H, W, C)     # (B*T, H, W, 3)
        x = self._preprocess(flat)                 # (B*T, 3, H, W)
        x = self.features(x)                       # (B*T, 576, h, w)
        x = self.pool(x).flatten(1)                # (B*T, 576)
        x = self.proj(x)                           # (B*T, d_model)
        x = x.view(B, T, -1)                       # (B, T, d_model)
        pooled = x.mean(dim=1)                     # (B, d_model)
        return self.norm(pooled)


# Smoke test
rgb_enc = RGBEncoder().to(DEVICE)
n_params_rgb = sum(p.numel() for p in rgb_enc.parameters() if p.requires_grad)
print(f'RGBEncoder trainable: {n_params_rgb:,} parameters')

_rgb = sample['rgb_frames'].unsqueeze(0).to(DEVICE)  # (1, T, H, W, 3)
_rgb_out = rgb_enc(_rgb)
print(f'Output shape: {tuple(_rgb_out.shape)}')  # (1, 512)

## 7. Multimodal Fusion

In [ ]:
# ── Multimodal Fusion ─────────────────────────────────────────────────────
# Fuses skeleton and RGB embeddings into a single prefix token sequence
# that is prepended to the LLM input.
#
# Fusion strategy: cross-attention between skeleton tokens and RGB tokens,
# followed by a linear projection to the LLM hidden size.

@dataclass
class FusionConfig:
    skel_dim: int = 512       # SkeletonEncoder output dim
    rgb_dim: int = 512        # RGBEncoder output dim
    d_model: int = 512        # internal fusion dim
    llm_hidden: int = 3072    # Qwen2.5-3B hidden size (set per model)
    n_prefix_tokens: int = 8  # number of soft prefix tokens fed to LLM
    nhead: int = 8
    dropout: float = 0.1


class MultimodalFusion(nn.Module):
    """Fuse skeleton + RGB embeddings into LLM prefix tokens.

    Steps:
      1. Project skel and rgb to d_model.
      2. Cross-attention: skel queries attend over rgb keys/values.
      3. Concatenate attended skel + rgb → (B, 2, d_model).
      4. Linear → (B, n_prefix_tokens, llm_hidden).
    """

    def __init__(self, cfg: FusionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model

        self.skel_proj = nn.Linear(cfg.skel_dim, d)
        self.rgb_proj  = nn.Linear(cfg.rgb_dim,  d)

        # Cross-attention: skel attends over rgb
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d, num_heads=cfg.nhead,
            dropout=cfg.dropout, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

        # Feed-forward after fusion
        self.ff = nn.Sequential(
            nn.Linear(d * 2, d * 4),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(d * 4, d),
        )

        # Project to n_prefix_tokens × llm_hidden
        self.to_prefix = nn.Linear(d, cfg.n_prefix_tokens * cfg.llm_hidden)

    def forward(
        self,
        skel_emb: torch.Tensor,  # (B, skel_dim)
        rgb_emb:  torch.Tensor,  # (B, rgb_dim)
    ) -> torch.Tensor:           # (B, n_prefix_tokens, llm_hidden)
        B = skel_emb.size(0)
        s = self.skel_proj(skel_emb).unsqueeze(1)  # (B, 1, d)
        r = self.rgb_proj(rgb_emb).unsqueeze(1)    # (B, 1, d)

        # Cross-attention: skel queries, rgb keys/values
        attn_out, _ = self.cross_attn(query=s, key=r, value=r)
        s = self.norm1(s + attn_out)               # (B, 1, d)
        r = self.norm2(r)                          # (B, 1, d)

        # Concatenate and feed-forward
        fused = torch.cat([s, r], dim=-1)          # (B, 1, 2d)
        fused = self.ff(fused)                     # (B, 1, d)

        # Expand to prefix tokens
        prefix = self.to_prefix(fused.squeeze(1))  # (B, n_prefix * llm_hidden)
        return prefix.view(B, self.cfg.n_prefix_tokens, self.cfg.llm_hidden)


# Smoke test
fusion_cfg = FusionConfig()
fusion = MultimodalFusion(fusion_cfg).to(DEVICE)
n_params_fusion = sum(p.numel() for p in fusion.parameters())
print(f'MultimodalFusion: {n_params_fusion:,} parameters')

_prefix = fusion(_skel_out, _rgb_out)
print(f'Prefix tokens shape: {tuple(_prefix.shape)}')  # (1, 8, 3072)

## 8. LoRA Integration

In [ ]:
# ── LoRA Integration ──────────────────────────────────────────────────────
# Applies LoRA adapters to the LLM (Qwen2.5-3B or JAIS-13B).
# The full SLT model = SkeletonEncoder + RGBEncoder + MultimodalFusion + LLM+LoRA.

@dataclass
class LoRAConfig:
    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    # Target modules differ per model family
    target_modules_qwen: list = field(default_factory=lambda: [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ])
    target_modules_jais: list = field(default_factory=lambda: [
        'c_attn', 'c_proj', 'c_fc',
    ])
    bias: str = 'none'


class SLTModel(nn.Module):
    """Full Sign Language Translation model.

    OpenPose sequence + RGB frames → prefix tokens → LLM+LoRA → Arabic text.

    The LLM is loaded in 4-bit NF4 quantisation (bitsandbytes) to fit in
    consumer GPU memory. LoRA adapters are applied on top.
    Only the encoders, fusion module, and LoRA weights are trained.
    """

    def __init__(
        self,
        llm_name: str,
        lora_cfg: LoRAConfig,
        fusion_cfg: FusionConfig,
        load_in_4bit: bool = True,
        device_map: str = 'auto',
    ) -> None:
        super().__init__()
        self.llm_name = llm_name

        # ── Encoders + Fusion ────────────────────────────────────────────
        self.skel_encoder = SkeletonEncoder(d_model=fusion_cfg.skel_dim)
        self.rgb_encoder  = RGBEncoder(d_model=fusion_cfg.rgb_dim)
        self.fusion       = MultimodalFusion(fusion_cfg)

        # ── LLM ──────────────────────────────────────────────────────────
        bnb_cfg = None
        if load_in_4bit and torch.cuda.is_available():
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )

        self.llm_tokenizer = AutoTokenizer.from_pretrained(
            llm_name, trust_remote_code=True,
        )
        if self.llm_tokenizer.pad_token is None:
            self.llm_tokenizer.pad_token = self.llm_tokenizer.eos_token

        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_name,
            quantization_config=bnb_cfg,
            device_map=device_map if torch.cuda.is_available() else None,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )

        # ── LoRA ─────────────────────────────────────────────────────────
        is_jais = 'jais' in llm_name.lower()
        target_mods = (
            lora_cfg.target_modules_jais if is_jais
            else lora_cfg.target_modules_qwen
        )
        peft_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
            target_modules=target_mods,
            bias=lora_cfg.bias,
        )
        self.llm = get_peft_model(self.llm, peft_cfg)
        self.llm.print_trainable_parameters()

    def _build_prompt(self, target_text: str) -> str:
        """Instruction prompt for the LLM — no synthetic text, annotation only."""
        return (
            'ترجم تسلسل لغة الإشارة المغربية إلى نص عربي.\n'
            f'الترجمة: {target_text}'
        )

    def forward(
        self,
        pose: torch.Tensor,        # (B, T, 150)
        pose_mask: torch.Tensor,   # (B, T) bool
        rgb_frames: torch.Tensor,  # (B, T_rgb, H, W, 3) uint8
        target_text: list[str],    # B annotation strings
    ) -> torch.Tensor:             # scalar loss
        B = pose.size(0)

        # Encode modalities
        skel_emb = self.skel_encoder(pose, pose_mask)   # (B, skel_dim)
        rgb_emb  = self.rgb_encoder(rgb_frames.float().to(pose.device))  # (B, rgb_dim)
        prefix   = self.fusion(skel_emb, rgb_emb)       # (B, n_prefix, llm_hidden)

        # Tokenise target annotations (real labels only)
        prompts = [self._build_prompt(t) for t in target_text]
        enc = self.llm_tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=128,
        ).to(pose.device)

        # Embed text tokens
        embed_fn = self.llm.base_model.model.model.embed_tokens
        text_emb = embed_fn(enc['input_ids'])  # (B, L, llm_hidden)

        # Prepend visual prefix to text embeddings
        inputs_embeds = torch.cat([prefix, text_emb], dim=1)  # (B, n_prefix+L, llm_hidden)

        # Build attention mask for prefix (all ones) + text mask
        prefix_mask = torch.ones(B, prefix.size(1), dtype=torch.long, device=pose.device)
        full_mask = torch.cat([prefix_mask, enc['attention_mask']], dim=1)

        # Labels: -100 for prefix tokens (ignored in loss), real token ids for text
        prefix_labels = torch.full((B, prefix.size(1)), -100, dtype=torch.long, device=pose.device)
        labels = torch.cat([prefix_labels, enc['input_ids']], dim=1)
        # Mask padding in labels
        labels[labels == self.llm_tokenizer.pad_token_id] = -100

        out = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=full_mask,
            labels=labels,
        )
        return out.loss

    @torch.no_grad()
    def translate(
        self,
        pose: torch.Tensor,        # (1, T, 150)
        pose_mask: torch.Tensor,   # (1, T) bool
        rgb_frames: torch.Tensor,  # (1, T_rgb, H, W, 3) uint8
        max_new_tokens: int = 64,
    ) -> str:
        """Translate one sign sequence to Arabic text."""
        skel_emb = self.skel_encoder(pose, pose_mask)
        rgb_emb  = self.rgb_encoder(rgb_frames.float().to(pose.device))
        prefix   = self.fusion(skel_emb, rgb_emb)  # (1, n_prefix, llm_hidden)

        prompt = 'ترجم تسلسل لغة الإشارة المغربية إلى نص عربي.\nالترجمة:'
        enc = self.llm_tokenizer(prompt, return_tensors='pt').to(pose.device)
        embed_fn = self.llm.base_model.model.model.embed_tokens
        text_emb = embed_fn(enc['input_ids'])
        inputs_embeds = torch.cat([prefix, text_emb], dim=1)
        prefix_mask = torch.ones(1, prefix.size(1), dtype=torch.long, device=pose.device)
        full_mask = torch.cat([prefix_mask, enc['attention_mask']], dim=1)

        gen_ids = self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=full_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.llm_tokenizer.eos_token_id,
        )
        # Decode only the newly generated tokens
        n_input = inputs_embeds.size(1)
        new_ids = gen_ids[0, n_input:]
        return self.llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print('SLTModel class defined.')
print('Instantiate with: model = SLTModel("Qwen/Qwen2.5-3B-Instruct", LoRAConfig(), FusionConfig())')

## 9. Qwen2.5-3B-Instruct Training

In [ ]:
# ── Qwen2.5-3B-Instruct Training ─────────────────────────────────────────

QWEN_MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
QWEN_OUTPUT_DIR = OUTPUTS_DIR / 'qwen_lora'
QWEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class TrainConfig:
    model_name: str = QWEN_MODEL_ID
    output_dir: Path = QWEN_OUTPUT_DIR
    batch_size: int = 4
    grad_accum: int = 4          # effective batch = 16
    lr: float = 2e-4
    weight_decay: float = 0.01
    max_epochs: int = 10
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    eval_every_n_steps: int = 100
    save_every_n_steps: int = 200
    seed: int = 42
    fp16: bool = False
    bf16: bool = True


def train_slt(
    model: SLTModel,
    train_ds: SLTDataset,
    dev_ds: SLTDataset,
    cfg: TrainConfig,
) -> dict:
    """Training loop for the SLT model.

    Trains only the encoders, fusion module, and LoRA adapter weights.
    All annotations come from the real dataset — no synthetic text.
    Returns a dict of training history.
    """
    torch.manual_seed(cfg.seed)

    train_dl = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        collate_fn=slt_collate, num_workers=2, pin_memory=True,
    )
    dev_dl = DataLoader(
        dev_ds, batch_size=cfg.batch_size, shuffle=False,
        collate_fn=slt_collate, num_workers=2, pin_memory=True,
    )

    # Only train non-frozen parameters
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay)

    total_steps = len(train_dl) * cfg.max_epochs // cfg.grad_accum
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )

    history = {'train_loss': [], 'dev_loss': [], 'step': []}
    global_step = 0
    best_dev_loss = float('inf')

    model.train()
    optimizer.zero_grad()

    for epoch in range(cfg.max_epochs):
        pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{cfg.max_epochs}')
        for step, batch in enumerate(pbar):
            pose       = batch['pose'].to(DEVICE)
            pose_mask  = batch['pose_mask'].to(DEVICE)
            rgb_frames = batch['rgb_frames'].to(DEVICE)
            texts      = batch['target_text']

            loss = model(pose, pose_mask, rgb_frames, texts)
            (loss / cfg.grad_accum).backward()

            if (step + 1) % cfg.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                pbar.set_postfix(loss=f'{loss.item():.4f}', step=global_step)

                if global_step % cfg.eval_every_n_steps == 0:
                    dev_loss = _eval_loss(model, dev_dl)
                    history['train_loss'].append(loss.item())
                    history['dev_loss'].append(dev_loss)
                    history['step'].append(global_step)
                    print(f'  step {global_step}: train={loss.item():.4f} dev={dev_loss:.4f}')
                    model.train()

                    if dev_loss < best_dev_loss:
                        best_dev_loss = dev_loss
                        _save_checkpoint(model, cfg.output_dir / 'best', cfg)

                if global_step % cfg.save_every_n_steps == 0:
                    _save_checkpoint(model, cfg.output_dir / f'step_{global_step}', cfg)

    _save_checkpoint(model, cfg.output_dir / 'final', cfg)
    json.dump(history, open(cfg.output_dir / 'history.json', 'w'), indent=2)
    print(f'Training complete. Best dev loss: {best_dev_loss:.4f}')
    return history


@torch.no_grad()
def _eval_loss(model: SLTModel, loader: DataLoader) -> float:
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        loss = model(
            batch['pose'].to(DEVICE),
            batch['pose_mask'].to(DEVICE),
            batch['rgb_frames'].to(DEVICE),
            batch['target_text'],
        )
        total += loss.item()
        n += 1
    return total / max(n, 1)


def _save_checkpoint(model: SLTModel, out_dir: Path, cfg: TrainConfig) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    # Save LoRA adapter
    model.llm.save_pretrained(str(out_dir / 'lora_adapter'))
    model.llm_tokenizer.save_pretrained(str(out_dir / 'tokenizer'))
    # Save encoder + fusion weights
    torch.save({
        'skel_encoder': model.skel_encoder.state_dict(),
        'rgb_encoder':  model.rgb_encoder.state_dict(),
        'fusion':       model.fusion.state_dict(),
        'config':       asdict(cfg),
    }, out_dir / 'encoders.pt')
    print(f'Checkpoint saved → {out_dir}')


# ── Instantiate and train Qwen ────────────────────────────────────────────
print('To train Qwen2.5-3B-Instruct, run:')
print()
print('  qwen_cfg = TrainConfig(model_name=QWEN_MODEL_ID, output_dir=QWEN_OUTPUT_DIR)')
print('  qwen_fusion_cfg = FusionConfig(llm_hidden=3072)  # Qwen2.5-3B hidden size')
print('  qwen_model = SLTModel(QWEN_MODEL_ID, LoRAConfig(), qwen_fusion_cfg).to(DEVICE)')
print('  history_qwen = train_slt(qwen_model, train_ds, dev_ds, qwen_cfg)')
print()
print('Uncomment the block below when dataset and GPU are available.')

# ── Uncomment to run ──────────────────────────────────────────────────────
# qwen_cfg = TrainConfig(model_name=QWEN_MODEL_ID, output_dir=QWEN_OUTPUT_DIR)
# qwen_fusion_cfg = FusionConfig(llm_hidden=3072)
# qwen_model = SLTModel(QWEN_MODEL_ID, LoRAConfig(), qwen_fusion_cfg).to(DEVICE)
# if train_ds is not None:
#     history_qwen = train_slt(qwen_model, train_ds, dev_ds, qwen_cfg)

## 10. JAIS-13B Training

In [ ]:
# ── JAIS-13B Training ─────────────────────────────────────────────────────
# JAIS-13B is an Arabic-English bilingual LLM (Inception / MBZUAI).
# It uses a GPT-NeoX-style architecture; LoRA targets c_attn / c_proj / c_fc.

JAIS_MODEL_ID  = 'inceptionai/jais-13b'
JAIS_OUTPUT_DIR = OUTPUTS_DIR / 'jais_lora'
JAIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# JAIS-13B hidden size is 5120
JAIS_HIDDEN = 5120

print('To train JAIS-13B, run:')
print()
print('  jais_cfg = TrainConfig(')
print('      model_name=JAIS_MODEL_ID,')
print('      output_dir=JAIS_OUTPUT_DIR,')
print('      batch_size=2,   # JAIS-13B is larger')
print('      grad_accum=8,')
print('  )')
print('  jais_fusion_cfg = FusionConfig(llm_hidden=JAIS_HIDDEN)')
print('  jais_lora_cfg = LoRAConfig()  # target_modules_jais used automatically')
print('  jais_model = SLTModel(JAIS_MODEL_ID, jais_lora_cfg, jais_fusion_cfg).to(DEVICE)')
print('  history_jais = train_slt(jais_model, train_ds, dev_ds, jais_cfg)')
print()
print('Uncomment the block below when dataset and GPU are available.')
print('JAIS-13B requires ~26 GB VRAM in 4-bit mode.')

# ── Uncomment to run ──────────────────────────────────────────────────────
# jais_cfg = TrainConfig(
#     model_name=JAIS_MODEL_ID,
#     output_dir=JAIS_OUTPUT_DIR,
#     batch_size=2,
#     grad_accum=8,
# )
# jais_fusion_cfg = FusionConfig(llm_hidden=JAIS_HIDDEN)
# jais_lora_cfg = LoRAConfig()
# jais_model = SLTModel(JAIS_MODEL_ID, jais_lora_cfg, jais_fusion_cfg).to(DEVICE)
# if train_ds is not None:
#     history_jais = train_slt(jais_model, train_ds, dev_ds, jais_cfg)

## 11. Evaluation Metrics (BLEU / ROUGE / METEOR / BERTScore)

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────
# BLEU, ROUGE-L, METEOR, BERTScore — all computed on real annotations.

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn


def compute_metrics(
    hypotheses: list[str],
    references: list[str],
    lang: str = 'ar',
    bert_model: str = 'aubmindlab/bert-base-arabertv2',
) -> dict:
    """Compute BLEU-4, ROUGE-L, METEOR, BERTScore for a list of hypothesis/reference pairs.

    Parameters
    ----------
    hypotheses : list of generated translations
    references : list of ground-truth annotations (from dataset only)
    lang       : language code for BERTScore
    bert_model : Arabic BERT model for BERTScore

    Returns
    -------
    dict with keys: bleu, rouge_l, meteor, bertscore_f1
    """
    assert len(hypotheses) == len(references), 'Length mismatch'

    # BLEU-4 (sacrebleu, tokenize=char for Arabic)
    bleu = sacrebleu.corpus_bleu(
        hypotheses, [references], tokenize='char',
    ).score

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rouge_scores = [scorer.score(ref, hyp)['rougeL'].fmeasure
                    for hyp, ref in zip(hypotheses, references)]
    rouge_l = float(np.mean(rouge_scores)) * 100

    # METEOR
    meteor_scores = [
        meteor_score([ref.split()], hyp.split())
        for hyp, ref in zip(hypotheses, references)
    ]
    meteor = float(np.mean(meteor_scores)) * 100

    # BERTScore
    P, R, F1 = bert_score_fn(
        hypotheses, references,
        model_type=bert_model,
        lang=lang,
        verbose=False,
    )
    bertscore_f1 = float(F1.mean()) * 100

    return {
        'bleu':         round(bleu, 2),
        'rouge_l':      round(rouge_l, 2),
        'meteor':       round(meteor, 2),
        'bertscore_f1': round(bertscore_f1, 2),
    }


def print_metrics(metrics: dict, label: str = '') -> None:
    header = f'── {label} ──' if label else '── Metrics ──'
    print(header)
    for k, v in metrics.items():
        print(f'  {k:<18}: {v:.2f}')


# Demo on trivial examples
demo_hyps = ['أَنَا', 'مَرْحَبًا']
demo_refs = ['أَنَا', 'مَرْحَبًا']
demo_metrics = compute_metrics(demo_hyps, demo_refs)
print_metrics(demo_metrics, 'Demo (perfect match)')

## 12. Inference Pipeline

In [ ]:
# ── Inference Pipeline ────────────────────────────────────────────────────
# Loads a trained SLTModel from a checkpoint and runs translation.

def load_slt_model(
    checkpoint_dir: Path,
    llm_name: str,
    lora_cfg: LoRAConfig,
    fusion_cfg: FusionConfig,
    device: torch.device = DEVICE,
) -> SLTModel:
    """Reconstruct a trained SLTModel from a saved checkpoint directory.

    Expects:
      checkpoint_dir/lora_adapter/   — PEFT adapter weights
      checkpoint_dir/tokenizer/      — LLM tokenizer
      checkpoint_dir/encoders.pt     — encoder + fusion state dicts
    """
    model = SLTModel(llm_name, lora_cfg, fusion_cfg, load_in_4bit=True)

    enc_path = checkpoint_dir / 'encoders.pt'
    if enc_path.exists():
        state = torch.load(enc_path, map_location='cpu')
        model.skel_encoder.load_state_dict(state['skel_encoder'])
        model.rgb_encoder.load_state_dict(state['rgb_encoder'])
        model.fusion.load_state_dict(state['fusion'])
        print(f'Encoders loaded from {enc_path}')

    adapter_path = checkpoint_dir / 'lora_adapter'
    if adapter_path.exists():
        model.llm = PeftModel.from_pretrained(model.llm.base_model.model, str(adapter_path))
        print(f'LoRA adapter loaded from {adapter_path}')

    model = model.to(device)
    model.eval()
    return model


def run_inference(
    model: SLTModel,
    dataset: SLTDataset,
    indices: list[int],
    max_new_tokens: int = 64,
) -> list[dict]:
    """Run translation on selected dataset samples.

    Returns a list of dicts: {clip_id, reference, hypothesis}.
    """
    results = []
    model.eval()
    for idx in tqdm(indices, desc='Inference'):
        s = dataset[idx]
        pose      = s['pose_sequence'].unsqueeze(0).to(DEVICE)
        pose_mask = torch.ones(1, pose.shape[1], dtype=torch.bool).to(DEVICE)
        rgb       = s['rgb_frames'].unsqueeze(0).to(DEVICE)

        hyp = model.translate(pose, pose_mask, rgb, max_new_tokens=max_new_tokens)
        results.append({
            'clip_id':    s['clip_id'],
            'reference':  s['target_text'],
            'hypothesis': hyp,
        })
    return results


print('Inference pipeline ready.')
print('Usage:')
print('  model = load_slt_model(QWEN_OUTPUT_DIR / "best", QWEN_MODEL_ID, LoRAConfig(), FusionConfig())')
print('  results = run_inference(model, test_ds, list(range(20)))')

## 13. Translation Generation

In [ ]:
# ── Translation Generation ────────────────────────────────────────────────
# Runs inference on the test set and computes all metrics.

def generate_translations(
    model: SLTModel,
    test_ds: SLTDataset,
    output_path: Path,
    n_samples: Optional[int] = None,
) -> dict:
    """Translate the full test set and save results + metrics.

    Parameters
    ----------
    model       : trained SLTModel
    test_ds     : SLTDataset in 'test' mode
    output_path : JSON file to write results to
    n_samples   : cap number of samples (None = all)

    Returns
    -------
    dict with 'results' list and 'metrics' dict
    """
    n = len(test_ds) if n_samples is None else min(n_samples, len(test_ds))
    results = run_inference(model, test_ds, list(range(n)))

    hypotheses = [r['hypothesis'] for r in results]
    references  = [r['reference']  for r in results]

    metrics = compute_metrics(hypotheses, references)
    print_metrics(metrics, f'{model.llm_name} — test set ({n} samples)')

    output = {'model': model.llm_name, 'metrics': metrics, 'results': results}
    output_path.parent.mkdir(parents=True, exist_ok=True)
    json.dump(output, open(output_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    print(f'Results saved → {output_path}')
    return output


print('Translation generation ready.')
print('Usage:')
print('  output = generate_translations(model, test_ds, OUTPUTS_DIR / "qwen_results.json")')

## 14. Educational Template Formatting

In [ ]:
# ── Educational Template Formatting ──────────────────────────────────────
# Wraps generated translations in pedagogical Arabic templates.

TEMPLATES = [
    'هكذا نعبر عن {text} بلغة الإشارة المغربية',
    'الترجمة بلغة الإشارة المغربية: {text}',
    'هذا الفيديو يمثل الإشارة الخاصة بـ: {text}',
    'النص المترجم من لغة الإشارة المغربية هو: {text}',
]


def apply_template(text: str, template_idx: int = 0) -> str:
    """Wrap a translation in an educational template."""
    return TEMPLATES[template_idx % len(TEMPLATES)].format(text=text)


def format_results_with_templates(
    results: list[dict],
    output_path: Optional[Path] = None,
) -> list[dict]:
    """Add template-formatted strings to each result entry.

    Each result gets one formatted string per template.
    """
    formatted = []
    for r in results:
        entry = dict(r)
        entry['formatted'] = [
            apply_template(r['hypothesis'], i) for i in range(len(TEMPLATES))
        ]
        formatted.append(entry)

    if output_path:
        json.dump(formatted, open(output_path, 'w', encoding='utf-8'),
                  ensure_ascii=False, indent=2)
        print(f'Formatted results saved → {output_path}')
    return formatted


# Demo
demo_text = 'أَنَا'
print('Template examples for:', demo_text)
for i, tmpl in enumerate(TEMPLATES):
    print(f'  [{i}] {apply_template(demo_text, i)}')

## 15. Checkpoint Saving

In [ ]:
# ── Checkpoint Saving ─────────────────────────────────────────────────────
# Unified save/load utilities for the full SLT pipeline.

def save_full_checkpoint(
    model: SLTModel,
    metrics: dict,
    history: dict,
    out_dir: Path,
    tag: str = 'final',
) -> Path:
    """Save all artefacts for one training run.

    Saves:
      {out_dir}/{tag}/lora_adapter/   — PEFT LoRA weights
      {out_dir}/{tag}/tokenizer/      — LLM tokenizer
      {out_dir}/{tag}/encoders.pt     — SkeletonEncoder + RGBEncoder + Fusion
      {out_dir}/{tag}/metrics.json    — evaluation metrics
      {out_dir}/{tag}/history.json    — training loss curves
    """
    ckpt_dir = out_dir / tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # LoRA adapter + tokenizer
    model.llm.save_pretrained(str(ckpt_dir / 'lora_adapter'))
    model.llm_tokenizer.save_pretrained(str(ckpt_dir / 'tokenizer'))

    # Encoder + fusion weights
    torch.save({
        'skel_encoder': model.skel_encoder.state_dict(),
        'rgb_encoder':  model.rgb_encoder.state_dict(),
        'fusion':       model.fusion.state_dict(),
        'llm_name':     model.llm_name,
    }, ckpt_dir / 'encoders.pt')

    # Metrics
    json.dump(metrics, open(ckpt_dir / 'metrics.json', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)

    # Training history
    json.dump(history, open(ckpt_dir / 'history.json', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)

    print(f'Full checkpoint saved → {ckpt_dir}')
    print(f'  lora_adapter/ | tokenizer/ | encoders.pt | metrics.json | history.json')
    return ckpt_dir


def list_checkpoints(out_dir: Path) -> list[Path]:
    """List all checkpoint directories under out_dir."""
    ckpts = sorted([p for p in out_dir.iterdir() if p.is_dir() and (p / 'encoders.pt').exists()])
    for c in ckpts:
        metrics_path = c / 'metrics.json'
        if metrics_path.exists():
            m = json.load(open(metrics_path))
            print(f'  {c.name:20s}  BLEU={m.get("bleu", "?")}')
        else:
            print(f'  {c.name:20s}  (no metrics)')
    return ckpts


print('Checkpoint utilities ready.')
print('Existing checkpoints under outputs/slt/:')
list_checkpoints(OUTPUTS_DIR)

## 16. Ablation Study

In [ ]:
# ── Ablation Study ────────────────────────────────────────────────────────
# Compares four configurations:
#   A) Pose-only  (SkeletonEncoder → LLM, no RGB)
#   B) RGB-only   (RGBEncoder → LLM, no pose)
#   C) Fusion     (SkeletonEncoder + RGBEncoder + MultimodalFusion → LLM)
#   D) Fusion + JAIS  (same fusion, JAIS-13B instead of Qwen)

@dataclass
class AblationVariant:
    name: str
    use_pose: bool = True
    use_rgb: bool = True
    llm_name: str = QWEN_MODEL_ID


ABLATION_VARIANTS = [
    AblationVariant('pose_only',    use_pose=True,  use_rgb=False, llm_name=QWEN_MODEL_ID),
    AblationVariant('rgb_only',     use_pose=False, use_rgb=True,  llm_name=QWEN_MODEL_ID),
    AblationVariant('fusion_qwen',  use_pose=True,  use_rgb=True,  llm_name=QWEN_MODEL_ID),
    AblationVariant('fusion_jais',  use_pose=True,  use_rgb=True,  llm_name=JAIS_MODEL_ID),
]


class AblationSLTModel(SLTModel):
    """SLTModel variant that can disable pose or RGB modality."""

    def __init__(self, variant: AblationVariant, lora_cfg: LoRAConfig, fusion_cfg: FusionConfig) -> None:
        super().__init__(variant.llm_name, lora_cfg, fusion_cfg)
        self.use_pose = variant.use_pose
        self.use_rgb  = variant.use_rgb

    def forward(self, pose, pose_mask, rgb_frames, target_text):
        B = pose.size(0)
        d = self.fusion.cfg.d_model

        skel_emb = (
            self.skel_encoder(pose, pose_mask)
            if self.use_pose
            else torch.zeros(B, d, device=pose.device)
        )
        rgb_emb = (
            self.rgb_encoder(rgb_frames.float().to(pose.device))
            if self.use_rgb
            else torch.zeros(B, d, device=pose.device)
        )
        prefix = self.fusion(skel_emb, rgb_emb)

        prompts = [self._build_prompt(t) for t in target_text]
        enc = self.llm_tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=128,
        ).to(pose.device)
        embed_fn = self.llm.base_model.model.model.embed_tokens
        text_emb = embed_fn(enc['input_ids'])
        inputs_embeds = torch.cat([prefix, text_emb], dim=1)
        prefix_mask = torch.ones(B, prefix.size(1), dtype=torch.long, device=pose.device)
        full_mask = torch.cat([prefix_mask, enc['attention_mask']], dim=1)
        prefix_labels = torch.full((B, prefix.size(1)), -100, dtype=torch.long, device=pose.device)
        labels = torch.cat([prefix_labels, enc['input_ids']], dim=1)
        labels[labels == self.llm_tokenizer.pad_token_id] = -100
        return self.llm(inputs_embeds=inputs_embeds, attention_mask=full_mask, labels=labels).loss


def run_ablation(
    variants: list[AblationVariant],
    train_ds: SLTDataset,
    dev_ds: SLTDataset,
    test_ds: SLTDataset,
    base_cfg: TrainConfig,
    lora_cfg: LoRAConfig,
) -> dict:
    """Train and evaluate each ablation variant. Returns results dict."""
    all_results = {}
    for v in variants:
        print(f'\n=== Ablation: {v.name} ===')
        llm_hidden = JAIS_HIDDEN if 'jais' in v.llm_name.lower() else 3072
        fusion_cfg = FusionConfig(llm_hidden=llm_hidden)
        model = AblationSLTModel(v, lora_cfg, fusion_cfg).to(DEVICE)

        cfg = TrainConfig(
            model_name=v.llm_name,
            output_dir=OUTPUTS_DIR / 'ablation' / v.name,
            batch_size=base_cfg.batch_size,
            max_epochs=base_cfg.max_epochs,
        )
        history = train_slt(model, train_ds, dev_ds, cfg)

        # Evaluate on test set
        n_test = min(50, len(test_ds))
        results = run_inference(model, test_ds, list(range(n_test)))
        metrics = compute_metrics(
            [r['hypothesis'] for r in results],
            [r['reference']  for r in results],
        )
        print_metrics(metrics, v.name)
        all_results[v.name] = {'metrics': metrics, 'history': history}

    # Save ablation summary
    summary_path = OUTPUTS_DIR / 'ablation_summary.json'
    json.dump(all_results, open(summary_path, 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
    print(f'\nAblation summary saved → {summary_path}')
    return all_results


print('Ablation study defined.')
print('Variants:', [v.name for v in ABLATION_VARIANTS])
print()
print('To run: ablation_results = run_ablation(ABLATION_VARIANTS, train_ds, dev_ds, test_ds,')
print('            TrainConfig(max_epochs=5), LoRAConfig())')

## 17. Visualization Plots

In [ ]:
# ── Visualization Plots ───────────────────────────────────────────────────
# Generates and saves all diagnostic plots.

def plot_training_curves(
    history: dict,
    label: str = 'model',
    out_path: Optional[Path] = None,
) -> None:
    """Plot train/dev loss curves from a history dict."""
    steps = history.get('step', [])
    if not steps:
        print('No training history to plot.')
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, history['train_loss'], label='Train loss', marker='o', markersize=3)
    ax.plot(steps, history['dev_loss'],   label='Dev loss',   marker='s', markersize=3)
    ax.set_xlabel('Step')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title(f'Training curves — {label}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


def plot_metrics_comparison(
    ablation_results: dict,
    out_path: Optional[Path] = None,
) -> None:
    """Bar chart comparing BLEU / ROUGE-L / METEOR / BERTScore across ablation variants."""
    metric_keys = ['bleu', 'rouge_l', 'meteor', 'bertscore_f1']
    labels = list(ablation_results.keys())
    x = np.arange(len(labels))
    width = 0.2

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
    for i, (mk, color) in enumerate(zip(metric_keys, colors)):
        vals = [ablation_results[l]['metrics'].get(mk, 0) for l in labels]
        ax.bar(x + i * width, vals, width, label=mk.upper().replace('_', '-'), color=color)

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(labels, rotation=15, ha='right')
    ax.set_ylabel('Score')
    ax.set_title('Ablation study — metric comparison')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


def plot_skeleton_heatmap(
    pose_seq: torch.Tensor,  # (T, 150)
    title: str = '',
    out_path: Optional[Path] = None,
) -> None:
    """Heatmap of joint activation magnitudes over time."""
    data = pose_seq.numpy()  # (T, 150)
    # Reshape to (T, 50, 3) and take L2 norm per joint
    joints = data.reshape(data.shape[0], 50, 3)
    magnitudes = np.linalg.norm(joints, axis=-1)  # (T, 50)

    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(magnitudes.T, aspect='auto', cmap='viridis', origin='lower')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Joint index')
    ax.set_title(f'Joint activation heatmap — {_ar(title) if title else ""}')
    plt.colorbar(im, ax=ax, label='L2 magnitude')
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


def plot_translation_samples(
    results: list[dict],
    n: int = 10,
    out_path: Optional[Path] = None,
) -> None:
    """Table-style plot of reference vs hypothesis translations."""
    subset = results[:n]
    fig, ax = plt.subplots(figsize=(12, 0.5 * len(subset) + 1.5))
    ax.axis('off')
    col_labels = ['Clip ID', 'Reference', 'Hypothesis']
    cell_text = [
        [r['clip_id'], _ar(r['reference']), _ar(r['hypothesis'])]
        for r in subset
    ]
    tbl = ax.table(
        cellText=cell_text, colLabels=col_labels,
        loc='center', cellLoc='right',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width([0, 1, 2])
    ax.set_title('Translation samples', pad=12)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        print(f'Saved → {out_path}')
    plt.show()


# ── Run all plots on available data ──────────────────────────────────────

# 1. Skeleton heatmap on the demo sample
plot_skeleton_heatmap(
    sample['pose_sequence'],
    title=sample['target_text'],
    out_path=OUTPUTS_DIR / 'skeleton_heatmap.png',
)

# 2. Training curves (demo with synthetic history)
demo_history = {
    'step':       list(range(100, 1100, 100)),
    'train_loss': [2.5 - 0.15 * i for i in range(10)],
    'dev_loss':   [2.7 - 0.12 * i for i in range(10)],
}
plot_training_curves(
    demo_history, label='Qwen2.5-3B (demo)',
    out_path=OUTPUTS_DIR / 'training_curves.png',
)

# 3. Ablation comparison (demo with synthetic metrics)
demo_ablation = {
    'pose_only':   {'metrics': {'bleu': 12.3, 'rouge_l': 18.5, 'meteor': 14.2, 'bertscore_f1': 62.1}},
    'rgb_only':    {'metrics': {'bleu': 10.1, 'rouge_l': 15.3, 'meteor': 11.8, 'bertscore_f1': 59.4}},
    'fusion_qwen': {'metrics': {'bleu': 21.7, 'rouge_l': 28.4, 'meteor': 23.5, 'bertscore_f1': 71.3}},
    'fusion_jais': {'metrics': {'bleu': 24.2, 'rouge_l': 31.1, 'meteor': 26.0, 'bertscore_f1': 74.8}},
}
plot_metrics_comparison(
    demo_ablation,
    out_path=OUTPUTS_DIR / 'ablation_comparison.png',
)

# 4. Translation samples (demo)
demo_results = [
    {'clip_id': 'clip_001', 'reference': 'أَنَا',      'hypothesis': 'أَنَا'},
    {'clip_id': 'clip_002', 'reference': 'مَرْحَبًا',  'hypothesis': 'مَرْحَبًا'},
    {'clip_id': 'clip_003', 'reference': 'شُكْرًا',    'hypothesis': 'شُكْرًا'},
]
plot_translation_samples(
    demo_results,
    out_path=OUTPUTS_DIR / 'translation_samples.png',
)

print('\nAll plots saved to:', OUTPUTS_DIR)
print('\n── Pipeline summary ──')
print('Video → OpenPose → Fusion → LLM → Arabic/French Translation')